# Task A — Adaptive Discrete Puzzle Solver

**Yandex ML Cup — ML track**

**Goal:** one universal algorithm that, given **50 min to train** and **25 min to solve**, cracks *any* reversible discrete puzzle through a single `gym.py` API — including hidden puzzles it has never seen.

**Final score: 82** (baseline started at ~63).

---

## The puzzles

| Puzzle | Move type | Structure |
|---|---|---|
| `game_15_2d` — 15-puzzle | `SWAP` + `EMPTY` | slide tiles into a blank |
| `toggle_lights` — Lights Out | `TOGGLE` | pressing a cell flips its row + column |
| `cylinder_game` — Vary­kon cylinder | `ROTATE` | rotate rings of a colored cylinder |
| *hidden* | unknown | same API, unknown mechanics |

The solver only ever talks to the environment through `reset`, `valid_actions`, `step`, `is_solved`, `encode_state`. It is **never told which puzzle it is playing** — that's what makes it "adaptive."

## The core idea — learn a value function `V(s)`

The baseline trains a small neural net `V(s)` that estimates **how many moves remain** until the puzzle is solved, then uses it as the heuristic in A* search (`f = g + V`).

**Where the training data comes from — backward walks.**  
Start from the *solved* state and take random legal moves. After `k` steps, that state is (at most) `k` moves from solved. Free labeled data, no human solutions needed:

```
solved ──move──▶ s1 (dist≈1) ──move──▶ s2 (dist≈2) ──▶ ... ──▶ sk (dist≈k)
```

**The network is puzzle-agnostic.** Each cell becomes a 15-number vector (3D position, content type/value, target type/value, match flags). Architecture: per-token MLP → mean+max pool → small head → `softplus`. Because it reads through `encode_state`, the *same weights* run on every puzzle regardless of grid size.

## Insight #1 — a learned `V` is useful, but never sufficient

The single biggest lesson from local experiments: **exact, structure-aware solvers beat learned search whenever they apply.** The ML value net is the *fallback*, not the star.

The clearest case is `toggle_lights`.

## Insight #2 — Lights Out is linear algebra over GF(2)

Each light is either on (1) or off (0). Pressing a cell flips a fixed set of lights — and pressing it **twice is the same as not pressing it** (mod 2). Order doesn't matter. So the whole puzzle is a system of linear equations over the field with two elements, **GF(2)**:

$$A \mathbf{x} = \mathbf{b} \pmod 2$$

- `A` — which lights each button toggles
- `b` — the current lit state
- `x` — which buttons to press (the solution)

Solve it with Gaussian elimination mod 2. This is **exact and instant** — it found the optimal solution on **20/20** instances (score 1.52) where beam search and A* both scored **0.0**.

In [ ]:
import numpy as np

def solve_gf2(A, b):
    """Gaussian elimination over GF(2). Returns x such that A @ x == b (mod 2)."""
    A = A.copy() % 2
    b = b.copy() % 2
    n_rows, n_cols = A.shape
    pivot_col_of_row = []
    row = 0
    for col in range(n_cols):
        # find a row at/below `row` with a 1 in this column
        piv = None
        for r in range(row, n_rows):
            if A[r, col]:
                piv = r
                break
        if piv is None:
            continue
        A[[row, piv]] = A[[piv, row]]          # swap into place
        b[[row, piv]] = b[[piv, row]]
        for r in range(n_rows):                 # eliminate the 1s elsewhere
            if r != row and A[r, col]:
                A[r] ^= A[row]                  # XOR == subtraction mod 2
                b[r] ^= b[row]
        pivot_col_of_row.append(col)
        row += 1

    x = np.zeros(n_cols, dtype=np.int8)
    for r, col in enumerate(pivot_col_of_row):
        x[col] = b[r]
    return x

# Tiny demo: 3 buttons, button i toggles lights i and i+1
A = np.array([[1,1,0],
              [0,1,1],
              [0,0,1]], dtype=np.int8)
b = np.array([1,0,1], dtype=np.int8)   # current lit state
x = solve_gf2(A, b)
print('press buttons:', x)
print('check A@x % 2 == b :', np.array_equal((A @ x) % 2, b))

## Insight #3 — beam search is the best *generic* fallback

For sliding-like and unknown puzzles, **beam search** with a smooth *mismatch* score (count of cells not in their target) beat everything else that generalizes:

| Puzzle | Method | Score | Solved |
|---|---|---|---|
| `game_15_2d` | beam + Manhattan/LC ranking | 0.0–0.19 | brittle, avoid |
| `game_15_2d` | **beam + mismatch ranking** | **0.51** | 4/8 |
| `game_15_2d` | fast array IDA* (EMPTY+SWAP) | 1.21 | 6/8 |
| `game_15_2d` | + retry reserve + dynamic margin | **1.39** | 7/8 |
| `cylinder_game` | greedy | 1.07 | 5/8 |
| `cylinder_game` | **long beam before A*** | **1.75** | 7/8 |

Counter-intuitive finding: **Manhattan + linear-conflict ranking *hurt* beam** — it collapsed diversity and the beam got stuck. A softer mismatch score kept the frontier varied and solved more.

## The solver cascade

The final `solve.py` runs a **structure-aware cascade** per instance — cheapest / most exact first, generic search last:

```
detect move structure via valid_actions / encode_state
         │
  is it pure TOGGLE?  ── yes ─▶  GF(2) exact solver          (optimal, instant)
         │ no
  is it EMPTY + SWAP? ── yes ─▶  fast-array IDA* (Manhattan+LC)
         │                        └─ leftover budget ─▶ beam (mismatch)
         │ no  (rotational / hidden)
         └────────────────────▶  long beam ─▶ A* with learned V  (fallback)
```

A reverse-BFS lookup table (built from the solved state at train time) provides optimal endings for any instance it covers.

## The bugs that actually moved the score

Most of the gain from 63 → 82 came from **fixing bugs**, not new models:

| Fix | Score |
|---|---|
| Start (baseline: beam + simple RL ranker) | ~63 |
| **beam ran only for sliding puzzles** — `if sol is None and sliding`. Cylinder & hidden puzzles only got A*, which without a good `V` solved almost nothing. Made beam the fallback for *all* types. | **63 → 70** |
| **BFS table `max_depth=6`** — test instances are random-walked 30–100 steps (optimal path 15–40). A depth-6 table covered ~0% of them. Raised to `max_depth=30`, `max_states=500_000`. | **70 → 82** |

Other correctness fixes:
- Hardcoded `CONTENT_NUM/CONTENT_EMPTY` instead of importing from `gym.py` → wrong for hidden puzzles.
- **Multiprocessing + PyTorch:** loading the model *before* `fork()` crashed workers. Fix: load `V` inside each worker *after* fork.
- Compact integer state keys instead of JSON in hot loops → ~1.5× faster (`7.14 µs → 4.39 µs` per state).

## Why performance engineering mattered as much as the algorithm

This is Python under a **hard 25-minute wall clock**. The heuristic *quality* barely changed the 15-puzzle solve rate — the *speed* of generating children did:

| Change | Throughput | 15-puzzle solved |
|---|---|---|
| gym-based `ordered_children` | ~40k children/s | 1/8 |
| fast list-swap + compact key | ~114k children/s | **6/8** |

Same heuristic, ~3× faster transitions → **6× more instances solved** in the same budget. Under a time limit, *states explored per second* is the real objective.

## Summary

1. **Exact structure-aware solvers first** — GF(2) for Lights Out is optimal and instant; no learned model can beat it.
2. **Learned `V(s)` is a fallback, not the core** — it beats uninformed A* but loses to exact solvers and often to beam.
3. **Generic beam search with smooth mismatch ranking** is the best all-rounder for unknown puzzles — diversity beats a sharp-but-brittle heuristic.
4. **Most of the score came from bugs and depth/budget tuning**, not architecture.
5. **Under a wall-clock budget, throughput is the objective** — fast native transitions solved 6× more than a fancier heuristic.

**Next idea (untested): table-guided beam / meet-in-the-middle** — beam goes forward ~12 steps, the reverse table covers ~20 from solved → combined coverage ~32, no extra training.